<img src="icon.png" width=128/>

# Allo: Accelerator Design and Programming Language

Lecture: SEU - FPGA HLS Design

Speaker: Kai Shao

Date: 2026/06/09

# 1 - Getting Started with Allo

In the lecture we traced the evolution of compilers as *a climb up the abstraction
ladder*: the more program structure an IR keeps, the more powerful the optimizations
it can express. The central thesis was

**"How" (how to compute) becomes a first-class citizen.**

Halide/TVM split a program into an **algorithm** (*what* to compute) and a
**schedule** (*how* to compute it). **Allo brings that idea to hardware**: it keeps
the **Payload IR** (the algorithm) and the **Transform IR** (the schedule) as two
separate, composable objects, and lowers them through MLIR down to synthesizable HLS C++.

This first notebook is pure onboarding. We will cover:

1. What Allo is and how to build it.
2. Writing your first kernel and running it.
3. Data types — including FPGA-flavoured **arbitrary-precision** integers.
4. The programming model: loops, memory, and the **SPMD** spatial model.
5. The basic user interface (inspecting IR, exporting to backends, templates).
6. Reading Allo's **error diagnostics**.

Scheduling, simulation, and synthesis each get their own notebook (2–4).

## 1.0 - Acknowledgements

The original Allo paper
- **Allo: A Programming Model for Composable Accelerator Design**, Hongzheng Chen, Niansong Zhang, Shaojie Xiang, Zhichen Zeng, Mengjia Dai, and Zhiru Zhang, Proc. ACM Program. Lang. 8, PLDI, Article 171 (June 2024), 2024. https://dl.acm.org/doi/10.1145/3656401
- **Dato: A Task-Based Programming Model for Dataflow Accelerators**, Shihan Fang, Hongzheng Chen, Niansong Zhang, Jiajie Li, Han Meng, Adrian Liu, Zhiru Zhang, arXiv:2509.06794, 2025. https://arxiv.org/abs/2509.06794

Upstream Allo provides a unified abstraction for accelerator design and
programming. Its core features include:

- Composable behavioral and structural design, so accelerator components can be
  built independently and assembled into larger systems.
- End-to-end deployment workflows that connect Python programs, PyTorch model
  flows, simulation, verification, and hardware code generation.
- An MLIR-based compiler stack with scheduling and transformation APIs for
  specializing kernels.
- Backend support for FPGA-oriented HLS flows and AI Engine targets, with CPU
  execution primarily used for functional validation.

Allo is open-source and available on GitHub: https://github.com/cornell-zhang/allo

This repository is a fork of upstream Allo. The active fork branch is `allov2`,
hosted at [kkkaishao/allo](https://github.com/kkkaishao/allo). It keeps the
upstream goal of composable accelerator design, while changing the frontend and
runtime interface to make Allo kernels feel more native in Python.

## 1.1 Where Allo sits

Recall the MLIR *lowering ladder* from the lecture:

```
   Tensor / Linalg     ← algebraic fusion, tiling, layout
   Affine / SCF        ← loop transforms, dependence-aware scheduling   ← Allo lives here
   LLVM dialect
   LLVM IR
   Machine code
```

Allo lets you **write the algorithm in Python**, captures it as MLIR at the
**affine/SCF** level (where the iteration space and memory accesses are still
explicit), and then applies an explicit, composable **schedule** before lowering to:

* a **CPU** target (a JIT'd shared library — for fast functional simulation), or
* a **Vitis HLS** target (synthesizable C++ for AMD/Xilinx FPGAs).

A key contrast with classic HLS: in Vivado/Vitis HLS you write `#pragma HLS pipeline`
*inside* the C source, tangling the schedule into the algorithm (this is what the
lecture called **destructive pragmas**). In Allo the schedule is a *separate* object,
so the algorithm is written once and many schedules can be applied and composed.

## 1.2 Building Allo

Allo's new (experimental) frontend lives in the `allo.exp` package and ships with an
MLIR/C++ backend, so it must be built once.

On Ubuntu or Debian:

```bash
sudo apt-get update
sudo apt-get install -y \
  build-essential \
  ccache \
  clang \
  cmake \
  lld \
  ninja-build \
  python3 \
  python3-dev \
  python3-pip \
  python3-venv
```

On macOS, install the common dependencies with Homebrew:

```bash
brew install cmake ninja python@3.12 ccache
```

### Clone this repository and build the project:

```bash
git clone https://github.com/kkkaishao/allo.git
cd allo
git checkout allov2
git submodule update --init --recursive --depth 1
```

### Create a Python Environment

Using conda:

```bash
conda create -n allo python=3.12
conda activate allo
```

### Build LLVM and MLIR

From the repository root:

```bash
cmake -S externals/llvm-project/llvm -B externals/llvm-project/build -G Ninja \
  -DCMAKE_BUILD_TYPE=Release \
  -DLLVM_ENABLE_PROJECTS="clang;mlir" \
  -DLLVM_ENABLE_RUNTIMES="openmp" \
  -DLLVM_TARGETS_TO_BUILD="Native" \
  -DLLVM_ENABLE_ASSERTIONS=ON \
  -DLLVM_INSTALL_UTILS=ON \
  -DLLVM_USE_CCACHE=ON \
  -DLLVM_USE_LINKER=lld

ninja -C externals/llvm-project/build
```

On macOS, use the same source and build directories, but use the runtime list
below and omit Linux-specific linker settings to use system `lld`:

```bash
cmake -S externals/llvm-project/llvm -B externals/llvm-project/build -G Ninja \
  -DCMAKE_BUILD_TYPE=Release \
  -DLLVM_ENABLE_PROJECTS="clang;mlir" \
  -DLLVM_ENABLE_RUNTIMES="compiler-rt;openmp" \
  -DLLVM_TARGETS_TO_BUILD="Native" \
  -DLLVM_ENABLE_ASSERTIONS=ON \
  -DLLVM_INSTALL_UTILS=ON \
  -DLLVM_USE_CCACHE=ON

ninja -C externals/llvm-project/build
```

### Build Allo

From the repository root:

```bash
python -m pip install -v -e .
```

The cell below just checks that the package imports and reports whether a Vitis HLS
toolchain is visible (needed only for notebooks 3–4).

In [ ]:
import numpy as np

import allo.exp as allo                       # range, grid, get_wid, get_nw, Stream, max/min ...
from allo.exp.lang.kernel import kernel       # the @kernel decorator
from allo.exp.lang.core import i32, f32       # data types

from allo.exp.backend.vitis.core import is_vitis_available
print("Allo imported OK")
print("Vitis HLS available:", is_vitis_available())

# In VSCode/Jupyter, route Allo's logs to plain text instead of a live spinner
# widget. The spinner can render as an empty Output() in VSCode, making csim /
# synthesis look like it never starts (it is actually running underneath).
import allo.exp.logging as _allo_log
from rich.console import Console as _Console
_allo_log.console = _Console(stderr=True, force_interactive=False)

### Import conventions used throughout

| import | gives you |
|---|---|
| `import allo.exp as allo` | `allo.range`, `allo.grid`, `allo.get_wid`, `allo.get_nw`, `allo.Stream`, `allo.max`, `allo.min` |
| `from allo.exp.lang.kernel import kernel` | the `@kernel` decorator (also `consteval`, `KernelOptions`) |
| `from allo.exp.lang.core import i32, f32, ...` | data types (`i32`, `u8`, `f32`, `APInt`, `index`, `Stream`, `Template`, ...) |

All of the new frontend lives under `allo.exp`. Anything in `allo.*` outside `allo.exp` belongs to the upstream frontend — don't mix them.

## 1.3 Your first kernel

A kernel is a normal Python function decorated with `@kernel`. Every argument carries
a **type annotation**: a scalar type like `i32`, or an array (memref) type like
`f32[16]`. Arrays are passed in and out by reference (HLS has no dynamic allocation
at the interface), so a kernel writes its result into an output array.

In [ ]:
@kernel
def vadd(A: f32[16], B: f32[16], C: f32[16]):
    for i in range(16):
        C[i] = A[i] + B[i]

# Lower the algorithm and JIT it for the CPU to run it like an ordinary function.
mod = vadd.schedule().export("cpu")

a = np.random.rand(16).astype(np.float32)
b = np.random.rand(16).astype(np.float32)
c = np.zeros(16, dtype=np.float32)
mod(a, b, c)

print("max abs error:", np.max(np.abs(c - (a + b))))
assert np.allclose(c, a + b)

What just happened:

* `vadd.schedule()` builds a **Schedule** — the handle through which all
  transformations are applied (here we apply none).
* `.export("cpu")` lowers the (un-scheduled) algorithm all the way to LLVM, JIT-compiles
  it, and returns a callable. This is the **fast functional simulator** (notebook 3).

We'll look at the generated IR and the HLS C++ in §1.6.

## 1.4 Data types

Hardware cares about **bit width**. Unlike a CPU compiler that rounds everything up to
8/16/32/64-bit registers, an HLS design uses exactly as many bits as it needs — fewer
bits means smaller, faster hardware. Allo exposes this directly.

### Integers — including arbitrary precision

* Named widths: `i8, i16, i32, i64`, unsigned `u1, u8, u16, u32, u64`, and many in
  between (`i2 … i16`, `u2 … u16`).
* `u1` is also the boolean type (aliased as `bool`).
* **Arbitrary width** via `APInt(width, signed=...)` — e.g. a 5-bit signed integer
  `APInt(5, signed=True)`, or a 256-bit accumulator `APInt(256, signed=False)`.

### Floats

* `f16, f32, f64`, plus `bf16` (bfloat16). Arbitrary formats via
  `APFloat(exponent_bits, mantissa_bits)`.

### Others

* `index` — the loop-index / address type (lowers to MLIR `index`).
* `constexpr` — a compile-time constant (folded away, like C++ `constexpr`).

In [ ]:
from allo.exp.lang.core import APInt, u8, index, constexpr

i5  = APInt(5, signed=True)     # values in [-16, 15]
u12 = APInt(12, signed=False)   # values in [0, 4095]

@kernel
def widths(a: i5[4], b: u8[4], c: i5[4]):
    for i in range(4):
        c[i] = a[i] + b[i]       # arithmetic respects the 5-bit result width

# Inspect how the types appear in the synthesizable HLS interface:
hls = widths.schedule().export("vitis").hls_code
print(hls.splitlines()[0])       # the function signature line(s)
for line in hls.splitlines():
    if "widths(" in line:
        print(line.strip())
        break

### Arbitrary precision is bit-accurate

The result of a 5-bit signed add **wraps modulo 2^5** and sign-extends — exactly as the
hardware would. The CPU simulator models this faithfully (more in notebook 3):

In [ ]:
@kernel
def acc(A: i5[6]) -> i5:
    s: i5 = 0
    for i in range(6):
        s = s + A[i]
    return s

A = np.array([15, 15, 1, 0, 0, 0], dtype=np.int8)   # 15+15+1 = 31  ->  wraps to -1
print("hardware result:", int(acc(A)), " (31 mod 32 = 31 -> sign-extended -> -1)")

### Bit-slicing

Because integers have an explicit width, you can read and write **bit ranges** — a
staple of HLS code (packing/unpacking words, masks, etc.). The slice width must be a
**compile-time constant**.

In [ ]:
from allo.exp.lang.core import u32

@kernel
def pack(x: u32, out: u32[1]):
    y: u32 = x
    y[0:4] = 5          # write the low nibble
    out[0] = y[4:8]     # read bits [4,8)  -> a 4-bit value

# The generated IR uses dedicated bit ops:
ir = pack.schedule().payload
print("IR of the pack kernel:\n", ir)
print("uses bit ops:", "allo.bit" in str(pack.schedule().payload))

## 1.5 The programming model

### Loops

* `range(n)` / `allo.range(n)` — a sequential loop. Inside a kernel both refer to
  Allo's loop, which lowers to `affine.for`.
* `allo.grid(m, n, ...)` — a multi-dimensional **parallel** loop nest; lowers to
  `affine.parallel`. Use it when the iterations are independent.
* When you intend to *schedule* a loop later, give it a name: `range(n, name="i")`
  (so the schedule can refer to it). More on this in notebook 2.

### Memory

* Kernel arguments `f32[M, N]` are arrays (memrefs) at the interface.
* Local buffers are declared with a bare annotation: `buf: i32[8]`, optionally with a
  list initializer `lut: i32[4] = [10, 20, 30, 40]`.

In [ ]:
@kernel
def scale(A: f32[4, 4], B: f32[4, 4]):
    for i, j in allo.grid(4, 4):     # independent iterations -> affine.parallel
        B[i, j] = A[i, j] * 2.0

ir = str(scale.schedule().payload)
print("IR of the parallel loop:\n", ir)
print("parallel loop in IR:", "affine.parallel" in ir)

### SPMD: the spatial programming model

This is the model that makes Allo a good fit for **dataflow / systolic** hardware.
Instead of one sequential program, you describe **one processing element (PE)**, and a
`mapping` replicates it across a grid. Each PE instance asks *"which PE am I?"* and acts
accordingly — this is **Single-Program, Multiple-Data (SPMD)**, the same model Triton
uses for GPU tiles (lecture, Slide 23).

* `@kernel(mapping=[P0, P1, ...])` — replicate the PE over a `P0 × P1 × ...` grid.
* `allo.get_wid(axis)` — this PE's coordinate along `axis` (its "worker id").
* `allo.get_nw(axis)` — the number of PEs along `axis`.
* `Stream[T]` — a FIFO channel between PEs; `.put(v)` / `.get()`. Streams can be laid
  out as a grid too (`Stream[T][P0, P1]`) so neighbours can forward data.

Here is a minimal SPMD example: 4 PEs, each writes its own id into a buffer.

In [ ]:
@kernel
def spmd_demo(out: i32[4]):
    P: constexpr = 4

    @kernel(mapping=[P])           # replicate `pe` across 4 instances
    def pe(buf: i32[4]):
        k = allo.get_wid(0)        # 0,1,2,3 -- which PE am I?
        buf[k] = k

    pe(out)

ir = str(spmd_demo.schedule().payload)
print("IR of the SPMD demo:\n", ir)
print("mapping recorded in IR:", "mapping=[4]" in ir)

The 2D output-stationary **systolic GEMM** — the running example of notebooks 3–4 — is
built exactly this way: a grid of PEs, each forwarding `A` to its right neighbour and
`B` to the one below through `Stream` FIFOs. The lecture's punchline was that *regular
systolic arrays are surprisingly hard in plain HLS* (you fight the tool's inference
engine), but with an explicit spatial schedule they become natural. We'll see that
payoff in notebook 4.

## 1.6 The basic user interface

Everything flows from a kernel through its **Schedule**:

```
kernel  --.schedule()-->  Schedule  --.export("cpu")-->   callable (simulate)
                                   \-.export("vitis")-->  Vitis backend (.hls_code, .synth())
```

### Inspecting the IR

`s.payload` is the MLIR module. Printing it shows the affine/SCF-level IR that Allo will
optimize and lower — this is the "structure-preserving" IR from the lecture.

In [ ]:
@kernel
def saxpy(a: f32, X: f32[8], Y: f32[8]):
    for i in range(8):
        Y[i] = a * X[i] + Y[i]

print(str(saxpy.schedule().payload))

### Viewing the generated HLS C++

`.export("vitis").hls_code` returns the synthesizable C++ — no toolchain required just
to *look* at it.

In [ ]:
print(saxpy.schedule().export("vitis").hls_code)

### Parametric kernels with `Template`

Hardware is often parameterised by size. A `Template` is a compile-time parameter
(a type or an integer) that you bind with `kernel[...]` before scheduling.

In [ ]:
from allo.exp.lang.core import Template

M, K, N = Template("M"), Template("K"), Template("N")

@kernel(M, K, N)
def gemm(A: f32[M, K], B: f32[K, N], C: f32[M, N]):
    for i in range(M):
        for j in range(N):
            for k in range(K):
                C[i, j] += A[i, k] * B[k, j]

# Bind the template to concrete sizes, then schedule/run as usual.
gemm16 = gemm[16, 16, 16]
mod = gemm16.schedule().export("cpu")

A = np.random.rand(16, 16).astype(np.float32)
B = np.random.rand(16, 16).astype(np.float32)
C = np.zeros((16, 16), dtype=np.float32)
mod(A, B, C)
assert np.allclose(C, A @ B, rtol=1e-4)
print("templated 16x16 GEMM matches numpy:", np.allclose(C, A @ B, rtol=1e-4))

## 1.7 Error diagnostics

Allo type-checks and compiles the kernel when you call `.schedule()` (or run it). On an
error it raises a `CompilationError` that points at the **exact source line and column**,
with a caret underline — much like a C compiler. Good diagnostics matter: catching a
mistake here is far cheaper than discovering it after a multi-minute synthesis run.

In [ ]:
from allo.exp.compiler.errors import CompilationError

@kernel
def oops(x: i32, out: i32[1]):
    out[0] = x + y          # `y` is not defined

try:
    oops.schedule()
except CompilationError as e:
    print(e.render(color=False))

A few more diagnostics worth recognising:

* **Bit-slice width must be constant** — a slice like `x[lo:hi]` with runtime `lo,hi`
  has an unknowable width, which has no hardware meaning.
* **Return values need an explicit annotation** — `def f(...) -> i32:`.
* **`return` inside a loop / nested `if` is unsupported** — the structured control flow
  Allo synthesizes has to stay analyzable.

In [ ]:
@kernel
def bad_slice(lo: i32, hi: i32, x: u32, out: u32[1]):
    out[0] = x[lo:hi]       # dynamic width -> no fixed hardware width

try:
    bad_slice.schedule()
except CompilationError as e:
    print(e.render(color=False))

## Recap

* Allo embeds an HLS DSL in Python and keeps **algorithm (Payload IR)** separate from
  **schedule (Transform IR)** — *How* is a first-class, composable object.
* Types are **bit-accurate**, including arbitrary-precision integers/floats and
  bit-slicing — essential for efficient hardware.
* The **SPMD** model (`mapping`, `get_wid`, `Stream`) describes spatial / systolic
  hardware naturally.
* The interface is uniform: `kernel.schedule().export("cpu" | "vitis")`, with
  `s.payload` for IR and `.hls_code` for generated C++.
* Diagnostics pinpoint source locations early.

**Next:** [`02_scheduling.ipynb`](02_scheduling.ipynb) — applying and composing
schedules, and seeing *the same algorithm* turn into *different hardware*.